# Travel profile demo
One profile → one segment → safe input checks.

Use this file after reopening the laptop. Run **A and B before the talk**, then **C–F on stage**. G and H show saved CI/failure evidence. I is for new input during Q&A.

**Mode is always shown:** cloud calls the Azure API; local starts a real API on this laptop. Local is a fallback, not a cloud deployment demonstration. Segment names are provisional.

## Before the talk
Open VS Code in WSL. Select **MLOP-PJ (WSL)**: `~/.venvs/mlop-pj/bin/python`.

- `MODE = "cloud"`: needs internet and existing Azure resources. No local Docker, Azure login, training or deployment is needed.
- `MODE = "local"`: needs the existing local bundle and API packages. B starts a local server. Keep this kernel open; use J to stop it.
- After shutdown or kernel restart, run A and B again. Files and the existing environment remain on disk. Unsaved Python variables do not.
- Leave the mode as cloud for the normal presentation. Do not silently switch to local or saved evidence after a failure.

The full Thai preparation and speaking guide is in `PROJECT_SUMMARY_TH.md`, section 7. Do not run the old training/deployment notebooks to prepare this demo.

In [ ]:
# A. Setup and mode
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "reports/golden-profiles.json").is_file())
assert sys.platform == "linux" and Path(sys.prefix).name == "mlop-pj", "Select the project WSL kernel."
for folder in [ROOT, ROOT / "output/presentation"]:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
from demo_support import Demo, panel

MODE = "cloud"  # Use "local" only for an explicit local fallback.
if "demo" in globals():
    demo.close()
demo = Demo(ROOT, MODE)
panel("DEMO SETUP", "Run B before presenting.", [("Mode", demo.label), ("Model", demo.version)])

In [ ]:
# B. Prepare and check the API
demo.prepare()

## On stage: C–F (about 90 seconds)
Keep the code folded and outputs visible. Explain the input and result, rather than reading JSON. Green 422 means the invalid-input check worked.

### 1. Show the input
One saved example with 24 category ratings. Six bars are shown; expand the full list if asked. This is not a new live user survey.

In [ ]:
# C. Input
CASE = 0  # Choose 0–4 for the five saved examples.
profile = demo.profile(CASE)
demo.show_input(profile)

### 2. Get the prediction
Point to the segment, uncertainty and version. Distance is not a confidence percentage. This example is also checked against its golden result.

In [ ]:
# D. Prediction
result = demo.predict(profile, case=CASE)

### 3. Try incomplete input
Send just one of the required 24 ratings. The expected response is **422**, not a crash.

In [ ]:
# E. Invalid input
invalid_status = demo.invalid()

### 4. Check batch 100
Repeat one example 100 times in one request. All outputs are checked. This is a batch contract check, not a throughput benchmark.

In [ ]:
# F. Batch
batch_status = demo.batch()

## Saved CI evidence — slide 7
These are completed runs from the same tested source. Open the links before the talk and log in to GitHub. Red tests and a skipped deploy are expected in the deliberate failure run.

In [ ]:
# G. CI evidence
demo.show_ci()

## Saved staging incident — slide 10
Show detection, the real alert and confirmed recovery. The first recovery attempt failed. This does not inject another fault or claim automatic rollback.

In [ ]:
# H. Failure and recovery evidence
demo.show_incident()

## Optional: new input during Q&A
Change a rating below, within 0–5, then rerun I. Keep all 24 fields. This is modified input, so its segment is not compared with the original golden result. An uncertain valid profile may return no segment name.

In [ ]:
# I. Custom input
custom_profile = profile.copy()
custom_profile["Category 1"] = 2.5
demo.show_input(custom_profile)
custom_result = demo.predict(custom_profile)

## Finish
J stops the local process started by this notebook. It does not delete Azure resources. After the final live demo, follow the separate verified cloud cleanup steps in `docs/FINAL_STEPS.md`.

In [ ]:
# J. Stop this demo session
demo.close()
panel("DEMO SESSION CLOSED", "Local demo process stopped. Follow FINAL_STEPS for Azure cleanup.")